# Đánh giá E5 + Reranker Pipeline (TREC-style)

## Mục tiêu

1. **Precision@10, Recall@10, F1@10** với reranking
2. **Ngưỡng tối ưu** (EER = điểm FPR = FNR)
3. **Cải thiện Precision@10**
4. **Tìm n và k tối ưu**

## Data Structure (TREC-style)

```
queries.jsonl (INPUT)     -> {"qid": "q001", "query": "đặt mua bộ vest"}
qrels.jsonl (GT)         -> {"qid": "q001", "product_id": "B093B1PR53", "relevance": 1}
ecommerce.csv (CORPUS)    -> product_id, title, searchable_text, ...
```

In [ ]:
## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" \
    "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm

In [ ]:
## 2) Setup - Clone repo & Mount Drive

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")

if COLAB_REPO.exists() and (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only"], check=False)
elif not (COLAB_REPO / "embedding_project" / "data" / "ecommerce.csv").is_file():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)

REPO_DIR = COLAB_REPO
SCRIPTS = REPO_DIR / "embedding_project" / "scripts"
sys.path.insert(0, str(SCRIPTS))

from google.colab import drive
drive.mount("/content/drive")

print("Repo cloned/mounted successfully!")

In [ ]:
## 3) Kiểm tra file model và data

In [ ]:
import torch

# Model paths từ Google Drive
EMB_MODEL = Path("/content/drive/MyDrive/models/e5_base_finetuned_5000")
RERANKER = Path("/content/drive/MyDrive/models/reranker")

# Data paths
EVAL_CSV = REPO_DIR / "embedding_project/data/ecommerce.csv"
QUERIES_JSONL = REPO_DIR / "embedding_project/outputs/retrieval_eval/queries.jsonl"
QRELS_JSONL = REPO_DIR / "embedding_project/outputs/retrieval_eval/qrels.jsonl"

# Output
OUTPUT_JSON = REPO_DIR / "embedding_project/outputs/evaluation/trec_eval_results.json"

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check files
for label, p in [
    ("Embedding Model", EMB_MODEL),
    ("Reranker", RERANKER),
    ("Corpus CSV", EVAL_CSV),
    ("Queries", QUERIES_JSONL),
    ("Qrels", QRELS_JSONL),
]:
    status = "✅ OK" if p.exists() else "❌ MISSING"
    print(f"  {status} {label}: {p}")

In [ ]:
## 4) Nạp script đánh giá

In [ ]:
import importlib
import sys
from pathlib import Path

SCRIPT_PATH = SCRIPTS / "evaluate_reranker_trec.py"

if not SCRIPT_PATH.is_file():
    # Copy từ local nếu chưa có
    raise FileNotFoundError(f"Script not found: {SCRIPT_PATH}")

import evaluate_reranker_trec as trec
importlib.reload(trec)

print("Script loaded successfully!")

In [ ]:
## 5) Chạy đánh giá

In [ ]:
%%time

device = "cuda" if torch.cuda.is_available() else "cpu"

# Cấu hình đánh giá
N_VALUES = [10, 20, 30, 50, 75, 100]  # Các n để tìm recall
K_VALUES = [5, 10, 20, 30, 50]         # Các k để grid search
EVAL_K = 10                            # k chính để báo cáo
TARGET_RECALL = 0.95                   # Mục tiêu recall để chọn n
MAX_NEG_PER_QUERY = 5                  # Hard negatives cho threshold

result = trec.run_evaluation(
    embedding_model=EMB_MODEL,
    reranker_model=RERANKER,
    corpus_path=EVAL_CSV,
    queries_path=QUERIES_JSONL,
    qrels_path=QRELS_JSONL,
    output_path=OUTPUT_JSON,
    n_values=N_VALUES,
    k_values=K_VALUES,
    eval_k=EVAL_K,
    encode_batch=128,
    rerank_batch=32,
    max_neg_per_query=MAX_NEG_PER_QUERY,
    target_recall=TARGET_RECALL,
    device=device,
)

print("\n✅ Evaluation completed!")

In [ ]:
## 6) Hiển thị kết quả

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

EVAL_K = 10

result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))

print("=" * 70)
print("KẾT QUẢ ĐÁNH GIÁ E5 + RERANKER")
print("=" * 70)

print(f"\n📊 Dataset Info:")
print(f"   Corpus: {result['corpus_size']} products")
print(f"   Queries: {result['n_queries']}")
print(f"   Selected n: {result['selected_n']} (target recall={result['target_recall']})")

# Bi-encoder vs Reranker comparison
print(f"\n📈 Performance Comparison @{EVAL_K}:")
print("-" * 50)
bi = result['bi_encoder_only']
rer = result['reranker']

metrics_df = pd.DataFrame([
    {"Model": "Bi-encoder Only", 
     "P@10": bi['Precision@10'],
     "R@10": bi['Recall@10'],
     "F1@10": bi['F1@10'],
     "MRR@10": bi['MRR@10'],
     "NDCG@10": bi['NDCG@10']},
    {"Model": "Bi-encoder + Reranker", 
     "P@10": rer['Precision@10'],
     "R@10": rer['Recall@10'],
     "F1@10": rer['F1@10'],
     "MRR@10": rer['MRR@10'],
     "NDCG@10": rer['NDCG@10']},
])
metrics_df = metrics_df.set_index("Model")
display(metrics_df.style.format("{:.4f}"))

# Improvement
imp = result['improvement']
print(f"\n📈 Improvement:")
print(f"   Precision: +{imp['precision_delta']:.4f}")
print(f"   Recall:    +{imp['recall_delta']:.4f}")
print(f"   F1:        +{imp['f1_delta']:.4f}")

In [ ]:
## 7) Grid Search - Tìm k tối ưu

In [ ]:
# Grid search over k
grid = pd.DataFrame(result['grid_search_k'])
grid['k'] = grid['k'].astype(int)

print("=" * 60)
print("GRID SEARCH - TÌM k TỐI ƯU")
print("=" * 60)

grid_display = grid[['k', 'Precision@10', 'Recall@10', 'F1@10', 'MRR@10', 'NDCG@10']].copy()
grid_display.columns = ['k', 'P@10', 'R@10', 'F1@10', 'MRR@10', 'NDCG@10']
display(grid_display.style.format("{:.4f}"))

# Best k
best_k = result['optimal_k']['k']
best_f1 = result['optimal_k']['f1']
print(f"\n✅ Optimal k = {best_k} (F1@10 = {best_f1:.4f})")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Precision vs Recall
axes[0].plot(grid['k'], grid['Precision@10'], 'b-o', label='Precision@10')
axes[0].plot(grid['k'], grid['Recall@10'], 'r-o', label='Recall@10')
axes[0].plot(grid['k'], grid['F1@10'], 'g-o', label='F1@10')
axes[0].axvline(x=best_k, color='gray', linestyle='--', label=f'Optimal k={best_k}')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision/Recall/F1 vs k')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MRR vs NDCG
axes[1].plot(grid['k'], grid['MRR@10'], 'm-o', label='MRR@10')
axes[1].plot(grid['k'], grid['NDCG@10'], 'c-o', label='NDCG@10')
axes[1].axvline(x=best_k, color='gray', linestyle='--', label=f'Optimal k={best_k}')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
axes[1].set_title('Ranking Metrics vs k')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
## 8) Recall by n - Tìm n tối ưu

In [ ]:
# Recall by n
recall_data = result['recall_by_n']
recall_df = pd.DataFrame([
    {"n": n, "Recall@n": r} 
    for n, r in sorted(recall_data.items())
])

print("=" * 60)
print("RECALL BY n - TÌM n TỐI ƯU")
print("=" * 60)

display(recall_df.style.format({"Recall@n": "{:.4f}"}))

# Find minimum n for target recall
target = result['target_recall']
selected_n = result['selected_n']

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall_df['n'], recall_df['Recall@n'], 'b-o', linewidth=2, markersize=8)
ax.axhline(y=target, color='r', linestyle='--', label=f'Target Recall = {target}')
ax.axvline(x=selected_n, color='g', linestyle='--', label=f'Selected n = {selected_n}')
ax.fill_between(recall_df['n'], recall_df['Recall@n'], alpha=0.2)
ax.set_xlabel('n (Retrieval Pool Size)', fontsize=12)
ax.set_ylabel('Recall@n', fontsize=12)
ax.set_title('Recall vs n (Bi-encoder)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Selected n = {selected_n} (đạt Recall@{selected_n} >= {target})")

In [ ]:
## 9) Threshold Analysis - Ngưỡng tối ưu

In [ ]:
thr = result['threshold_analysis']
curve = pd.DataFrame(result['threshold_curve'])

print("=" * 70)
print("THRESHOLD ANALYSIS - NGƯỠNG TỐI ƯU")
print("=" * 70)

print(f"\n📊 Dataset for threshold:")
print(f"   Total pairs: {thr['n_pairs']}")
print(f"   Positive (relevant): {thr['n_positive']}")
print(f"   Negative (hard neg): {thr['n_negative']}")

# EER Analysis
eer = thr['EER']
print(f"\n🎯 EER Point (FPR ≈ FNR):")
print(f"   Threshold (τ): {eer['threshold']:.4f}")
print(f"   FPR: {eer['FPR']:.4f}")
print(f"   FNR: {eer['FNR']:.4f}")
print(f"   Error Rate: {eer['error_rate']:.4f}")

# Min Error Rate
me = thr['min_error']
print(f"\n📉 Min Error Rate Point:")
print(f"   Threshold (τ): {me['threshold']:.4f}")
print(f"   FPR: {me['FPR']:.4f}")
print(f"   FNR: {me['FNR']:.4f}")
print(f"   Error Rate: {me['error_rate']:.4f}")

# Optimal threshold
opt = thr['optimal']
print(f"\n✅ Optimal Threshold for Deployment:")
print(f"   τ = {opt['threshold']:.4f}")
print(f"   At this threshold: FPR = {opt['FPR']:.4f}, FNR = {opt['FNR']:.4f}")
print(f"   Total Error Rate = {opt['error_rate']:.4f}")

In [ ]:
# Plot FPR/FNR vs Threshold
curve['threshold'] = curve['threshold'].astype(float)
curve = curve.sort_values('threshold')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FPR and FNR vs Threshold
axes[0].plot(curve['threshold'], curve['FPR'], 'b-', label='FPR', linewidth=2)
axes[0].plot(curve['threshold'], curve['FNR'], 'r-', label='FNR', linewidth=2)
axes[0].axvline(x=eer['threshold'], color='green', linestyle='--', linewidth=2,
                label=f'EER τ={eer["threshold"]:.4f}')
axes[0].axvline(x=me['threshold'], color='orange', linestyle='--', linewidth=2,
                label=f'Min Error τ={me["threshold"]:.4f}')
axes[0].fill_between(curve['threshold'], curve['FPR'], alpha=0.2)
axes[0].fill_between(curve['threshold'], curve['FNR'], alpha=0.2)
axes[0].set_xlabel('Threshold (τ)', fontsize=12)
axes[0].set_ylabel('Rate', fontsize=12)
axes[0].set_title('FPR/FNR vs Threshold', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error Rate vs Threshold
axes[1].plot(curve['threshold'], curve['error_rate'], 'purple', linewidth=2)
axes[1].axvline(x=eer['threshold'], color='green', linestyle='--', linewidth=2,
                label=f'EER τ={eer["threshold"]:.4f}')
axes[1].axvline(x=me['threshold'], color='orange', linestyle='--', linewidth=2,
                label=f'Min Error τ={me["threshold"]:.4f}')
axes[1].fill_between(curve['threshold'], curve['error_rate'], alpha=0.2, color='purple')
axes[1].set_xlabel('Threshold (τ)', fontsize=12)
axes[1].set_ylabel('Error Rate', fontsize=12)
axes[1].set_title('Total Error Rate vs Threshold', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# DET curve (FPR vs FNR)
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(curve['FPR'], curve['FNR'], 'b-', linewidth=2)
ax.scatter([eer['FPR']], [eer['FNR']], color='red', s=200, zorder=5, 
           label=f'EER Point (τ={eer["threshold"]:.4f})')
ax.scatter([me['FPR']], [me['FNR']], color='orange', s=200, zorder=5, 
           label=f'Min Error Point (τ={me["threshold"]:.4f})')

# Perfect classifier line
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')

ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('False Negative Rate (FNR)', fontsize=12)
ax.set_title('DET Curve (Detection Error Tradeoff)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

In [ ]:
## 10) Chiến lược cải thiện Precision@10

In [ ]:
print("=" * 70)
print("CHIẾN LƯỢC CẢI THIỆN PRECISION@10")
print("=" * 70)

current_p = rer['Precision@10']
print(f"\n📍 Current Precision@10: {current_p:.4f}")

strategies = """
## Các chiến lược để cải thiện Precision@10:

### 1. **Tăng n (Retrieval Pool)**
   - Retriever lấy nhiều candidate hơn trước khi rerank
   - Cho reranker nhiều lựa chọn hơn để xếp hạng
   - Trade-off: Tăng latency và reranking cost

### 2. **Điều chỉnh ngưỡng reranker (τ)**
   - Dùng ngưỡng cao hơn để lọc bớt false positive
   - Chỉ hiển thị kết quả có score >= τ
   - Trade-off: Giảm recall, có thể miss relevant items

### 3. **Cascade Ranking**
   - Thêm model xếp hạng thứ 2 (lightweight)
   - 2-stage: retrieve → light rerank → heavy rerank
   - Giảm số lượng cần rerank bằng model nặng

### 4. **Hard Negative Mining**
   - Cải thiện training data cho reranker
   - Thêm các false positive khó vào training
   - Giúp reranker phân biệt tốt hơn

### 5. **Query Expansion / Rewriting**
   - Thêm context từ user's intent
   - Dùng LLM để paraphrase query
   - Cải thiện semantic matching

### 6. **Learning to Rank**
   - Train model với nhiều features hơn
   - Kết hợp text similarity + popularity + recency
   - LambdaMART, LightGBM với features
"""
print(strategies)

In [ ]:
# Experiment: Precision@10 với các k khác nhau
print("📊 Precision@10 với các k khác nhau (sau reranking):")
print("-" * 50)

precision_by_k = []
for entry in result['grid_search_k']:
    k = entry['k']
    p = entry.get('Precision@10', 0)
    r = entry.get('Recall@10', 0) if k <= 10 else entry.get(f'Recall@{k}', 0)
    f1 = entry.get('F1@10', 0) if k <= 10 else entry.get(f'F1@{k}', 0)
    precision_by_k.append({'k': k, 'P@10': p, 'R@10': r, 'F1@10': f1})

exp_df = pd.DataFrame(precision_by_k)
display(exp_df.style.format("{:.4f}"))

print("\n💡 Nhận xét: Khi k càng lớn, Precision@10 có xu hướng giảm vì")
print("   model phải chọn top-10 từ pool lớn hơn, khó hơn.")

In [ ]:
## 11) Tổng kết

In [ ]:
print("=" * 70)
print("TỔNG KẾT ĐÁNH GIÁ")
print("=" * 70)

summary = f"""
## 📊 Configuration
   - Corpus: {result['corpus_size']} products
   - Queries: {result['n_queries']}
   - Target Recall: {result['target_recall']}
   - Selected n: {result['selected_n']}

## 📈 Performance @{EVAL_K}
   | Metric      | Bi-encoder | +Reranker | Improvement |
   |-------------|------------|-----------|-------------|
   | Precision@10| {bi['Precision@10']:.4f}   | {rer['Precision@10']:.4f}    | +{imp['precision_delta']:.4f}     |
   | Recall@10   | {bi['Recall@10']:.4f}   | {rer['Recall@10']:.4f}    | +{imp['recall_delta']:.4f}     |
   | F1@10       | {bi['F1@10']:.4f}   | {rer['F1@10']:.4f}    | +{imp['f1_delta']:.4f}     |
   | MRR@10      | {bi['MRR@10']:.4f}   | {rer['MRR@10']:.4f}    | +{rer['MRR@10']-bi['MRR@10']:.4f}     |
   | NDCG@10     | {bi['NDCG@10']:.4f}   | {rer['NDCG@10']:.4f}    | +{rer['NDCG@10']-bi['NDCG@10']:.4f}     |

## 🎯 Optimal Parameters
   - Optimal k: {result['optimal_k']['k']} (F1@10 = {result['optimal_k']['f1']:.4f})
   - Selected n: {result['selected_n']} (Recall@{result['selected_n']} >= {result['target_recall']})

## ⚙️ Threshold for Deployment
   - EER Threshold (τ): {eer['threshold']:.4f}
   - FPR at EER: {eer['FPR']:.4f}
   - FNR at EER: {eer['FNR']:.4f}
   - Error Rate at EER: {eer['error_rate']:.4f}

## 📝 Deployment Recommendation
   Khi triển khai production:
   1. Dùng τ = {eer['threshold']:.4f} (EER point) để cân bằng FP và FN
   2. Hoặc τ = {me['threshold']:.4f} (min error) để tối thiểu hóa total error
   3. Lọc kết quả rerank có score >= τ trước khi trả về user
"""
print(summary)

# Save summary
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"\n✅ Results saved to: {OUTPUT_JSON}")